# retarget_body.ipynb — Tongue Body full-contour retargeting (수정용)

`retarget_body.py` 의 노트북 버전. 함수를 셀에 인라인해 **알고리즘·파라미터를 직접 고쳐가며** 확인.
순서대로 실행: 설정 → 지오메트리 → 모델 full contour 확인 → body 매핑·retarget → 4패널/슬라이더/GIF.
전제: `datasets/prepare.py`(png), `extract_body_contour.py`(body_contour.npz), `test/v8/<subject>/registration.csv` 준비됨.

In [ ]:
%matplotlib inline
import os, csv
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa
import imageio.v2 as imageio
from scipy.interpolate import RBFInterpolator
from scipy.ndimage import uniform_filter1d
from scipy.signal import savgol_filter
plt.rcParams["figure.dpi"] = 110

REPO = Path.cwd()
assert (REPO/"datasets").exists(), f"repo 루트에서 실행 (현재 {REPO})"

# ---- 여기만 바꾸면 됨 ----
SUBJECT   = "Subject3"
DATA      = REPO/"datasets"
N         = 60          # 닫힌 contour 리샘플 점수(=RBF center 수)
RBF_LEN   = 18.0        # skinning 길이 스케일(mm)
SPATIAL_WIN  = 3        # 곡선 따라(닫힘) 스무딩
TEMPORAL_WIN = 9        # 프레임축 Savitzky-Golay(홀수, 1=off)
TEMPORAL_POLY= 2
MM_PER_PX = 1.164
H = 256
REST = 0                # rest 프레임 인덱스(all_fr 기준)
print("config:", SUBJECT, "N", N, "RBF_LEN", RBF_LEN)

## 1. 지오메트리 헬퍼
`load_obj`, 모델 y=0 슬라이스 full contour, 닫힌 contour 정렬/리샘플, image→model affine.
**여기 함수를 고치면** 대응·contour 방식이 바뀝니다.

In [ ]:
def load_obj(path):
    V, F = [], []
    for L in open(path):
        t = L.split()
        if not t: continue
        if t[0] == "v": V.append([float(x) for x in t[1:4]])
        elif t[0] == "f": F.append([int(s.split("/")[0]) - 1 for s in t[1:4]])
    return np.array(V), np.array(F)

def align_resample(c, n):
    """닫힌 contour → tip(min x) 시작, dorsal-up 방향, 호길이 n점."""
    c = np.asarray(c, float)
    c = np.roll(c, -int(np.argmin(c[:, 0])), axis=0)        # tip = 최소 x
    if c[1, 1] < c[-1, 1]: c = np.roll(c[::-1], 1, axis=0)  # 다음 점이 위(z↑)로
    cc = np.vstack([c, c[:1]])
    d = np.r_[0, np.cumsum(np.hypot(np.diff(cc[:,0]), np.diff(cc[:,1])))]
    if d[-1] == 0: return None
    u = np.linspace(0, d[-1], n, endpoint=False)
    return np.column_stack([np.interp(u,d,cc[:,0]), np.interp(u,d,cc[:,1])])

def model_midsag_contour(V_mm, F, n):
    """tongue mesh 를 y=0 로 잘라 닫힌 midsag 외곽선 (x,z) mm, n점."""
    segs = []
    for tri in F:
        p = V_mm[tri]; s = p[:, 1]; pts = []
        for a, b in [(0,1),(1,2),(2,0)]:
            if (s[a] <= 0 < s[b]) or (s[b] <= 0 < s[a]):
                t = s[a]/(s[a]-s[b]); pts.append((p[a]+t*(p[b]-p[a]))[[0,2]])
        if len(pts) == 2: segs.append((pts[0], pts[1]))
    used=[False]*len(segs); poly=[np.array(segs[0][0]), np.array(segs[0][1])]; used[0]=True
    for _ in range(len(segs)):
        cur=poly[-1]; nxt=None
        for i,sg in enumerate(segs):
            if used[i]: continue
            a,b=np.array(sg[0]),np.array(sg[1])
            if np.hypot(*(a-cur))<0.5: nxt=(i,b); break
            if np.hypot(*(b-cur))<0.5: nxt=(i,a); break
        if nxt is None: break
        used[nxt[0]]=True; poly.append(nxt[1])
    return align_resample(np.array(poly), n)

def affine_image_to_model(reg_csv):
    img,mod=[],[]
    for r in csv.DictReader(open(reg_csv)):
        img.append([float(r["imageX"]),float(r["imageY"])]); mod.append([float(r["modelX"]),float(r["modelZ"])])
    img=np.array(img); mod=np.array(mod)
    A,*_=np.linalg.lstsq(np.column_stack([img,np.ones(len(img))]),mod,rcond=None)
    return lambda xy: np.column_stack([xy,np.ones(len(xy))])@A
print("helpers ready")

## 2. 모델 full contour 확인
tongue mesh 의 y=0 슬라이스가 깨끗한 닫힌 외곽선인지 눈으로 확인.

In [ ]:
V, F = load_obj(str(DATA/"tongue_model"/"tongue_rest_m.obj")); V = V*1000.0; V[:,0] += 2.0  # mm
mc = model_midsag_contour(V, F, N)                    # (N,2) 모델 full contour
plt.figure(figsize=(4.5,4.5))
plt.plot(np.r_[mc[:,0],mc[:1,0]], np.r_[mc[:,1],mc[:1,1]], "g-o", ms=3)
md_pts = V[np.abs(V[:,1])<6.0]
plt.scatter(md_pts[:,0], md_pts[:,2], s=6, c="0.8", zorder=0)
plt.gca().set_aspect("equal"); plt.title(f"model midsag full contour (N={N})"); plt.show()

## 3. Body contour → model 공간 → delta → RBF skinning
`body_contour.npz`(image space)를 registration affine 으로 model 공간으로 옮기고, 결측 프레임은
시간보간. delta 를 곡선/시간축 스무딩 후 Gaussian-RBF 로 mesh 변형.

In [ ]:
reg = str(REPO/"test"/"v8"/SUBJECT/"registration.csv")
sub = DATA/"MRI_SSFP_10fps"/SUBJECT
to_model = affine_image_to_model(reg)
npz = np.load(str(sub/"body_contour.npz"))
fids = [int(x) for x in npz["frame_ids"]]
all_fr = list(range(min(fids), max(fids)+1))

def body_model(mi):
    key = f"resamp_f{mi}"
    if key not in npz.files: return None
    rc = npz[key]
    xy = np.column_stack([rc[:,1]*MM_PER_PX, (H-1-rc[:,0])*MM_PER_PX])   # image-mm
    return align_resample(to_model(xy), N)

stack = np.full((len(all_fr), N, 2), np.nan)
for t,mi in enumerate(all_fr):
    b = body_model(mi)
    if b is not None: stack[t] = b
for c in range(2):                                    # 결측 프레임(예: f35) 시간보간
    for j in range(N):
        col=stack[:,j,c]; ok=~np.isnan(col)
        stack[:,j,c]=np.interp(np.arange(len(all_fr)), np.where(ok)[0], col[ok])

delta = stack - stack[REST]
if SPATIAL_WIN>1:  delta = uniform_filter1d(delta, SPATIAL_WIN, axis=1, mode="wrap")
if TEMPORAL_WIN>1 and len(all_fr)>=TEMPORAL_WIN: delta = savgol_filter(delta, TEMPORAL_WIN, TEMPORAL_POLY, axis=0, mode="interp")
delta = delta - delta[REST]

Vxz = V[:,[0,2]]; deformed=[]
for k in range(len(all_fr)):
    rbf = RBFInterpolator(mc, delta[k], kernel="gaussian", epsilon=1.0/RBF_LEN, degree=-1, smoothing=1e-3)
    dxz = rbf(Vxz); Vd=V.copy(); Vd[:,0]+=dxz[:,0]; Vd[:,2]+=dxz[:,1]; deformed.append(Vd)
gmax = max(np.linalg.norm(deformed[k]-V,axis=1).max() for k in range(len(all_fr)))
print(f"frames={len(all_fr)}  max disp={gmax:.1f}mm")

## 4. 단일 프레임 4패널 (수정하며 확인)
`show4(k)` — MRI+body / model midsag full / 3D mesh / 3D disp. 슬라이더로 넘겨보기.

In [ ]:
allv=np.vstack(deformed)
XL=(allv[:,0].min(),allv[:,0].max()); YL=(V[:,1].min(),V[:,1].max()); ZL=(allv[:,2].min(),allv[:,2].max())
rest_mid = V[np.abs(V[:,1])<6.0]

def show4(k, save=None):
    mi=all_fr[k]; Vd=deformed[k]; disp=np.linalg.norm(Vd-V,axis=1)
    fig=plt.figure(figsize=(15,4.4))
    a0=fig.add_subplot(1,4,1)
    g=imageio.imread(str(sub/"png"/f"image_{mi}.png")); a0.imshow(g,cmap="gray"); a0.axis("off"); a0.set_title(f"MRI f{mi}",fontsize=9)
    key=f"contour_f{mi}"
    if key in npz.files:
        c=npz[key]; cc=np.vstack([c,c[:1]]); a0.plot(cc[:,1],cc[:,0],"-",c="lime",lw=1.5)
        a0.set_xlim(c[:,1].min()-22,c[:,1].max()+22); a0.set_ylim(c[:,0].max()+22,c[:,0].min()-22)
    a1=fig.add_subplot(1,4,2); a1.scatter(rest_mid[:,0],rest_mid[:,2],s=5,c="0.8")
    mcd=mc+delta[k]; a1.plot(np.r_[mcd[:,0],mcd[:1,0]],np.r_[mcd[:,1],mcd[:1,1]],"b-",lw=1.4,label="model")
    bm=stack[k]; a1.plot(np.r_[bm[:,0],bm[:1,0]],np.r_[bm[:,1],bm[:1,1]],"r--",lw=1.2,label="MRI body")
    a1.set_aspect("equal",adjustable="box"); a1.set_xlim(XL); a1.set_ylim(ZL); a1.set_xticks([]); a1.set_yticks([])
    a1.set_title("model midsag full",fontsize=9)
    if k==0: a1.legend(fontsize=6,loc="lower left")
    a2=fig.add_subplot(1,4,3,projection="3d")
    a2.plot_trisurf(Vd[:,0],Vd[:,1],Vd[:,2],triangles=F,color="lightblue",alpha=0.9,linewidth=0.1,edgecolor="0.5")
    a2.set_xlim(XL); a2.set_ylim(YL); a2.set_zlim(ZL); a2.view_init(20,-70)
    a2.set_xticklabels([]); a2.set_yticklabels([]); a2.set_zticklabels([]); a2.set_title("3D mesh",fontsize=9)
    a3=fig.add_subplot(1,4,4,projection="3d")
    tp=a3.plot_trisurf(Vd[:,0],Vd[:,1],Vd[:,2],triangles=F,cmap="viridis",linewidth=0.1,edgecolor="0.4")
    tp.set_array(disp[F].mean(1)); tp.set_clim(0,gmax)
    a3.set_xlim(XL); a3.set_ylim(YL); a3.set_zlim(ZL); a3.view_init(20,-70)
    a3.set_xticklabels([]); a3.set_yticklabels([]); a3.set_zticklabels([]); a3.set_title(f"3D disp (max {disp.max():.1f}mm)",fontsize=9)
    fig.suptitle(f"Tongue-Body full-contour retargeting  f{mi}",fontsize=10)
    fig.subplots_adjust(left=0.01,right=0.99,top=0.86,bottom=0.03,wspace=0.06)
    if save: fig.savefig(save,dpi=95); plt.close(fig)
    else: plt.show()

try:
    from ipywidgets import interact, IntSlider
    interact(lambda k: show4(k), k=IntSlider(min=0,max=len(all_fr)-1,step=1,value=len(all_fr)//2))
except Exception as e:
    print("ipywidgets 없음 → show4(번호) 직접 호출:", e); show4(len(all_fr)//2)

## 5. 전체 GIF 저장
모든 프레임을 4패널로 묶어 `body_retarget_4panel.gif` 저장(3D 렌더라 다소 느림).

In [ ]:
OUT = str(sub/"body_retarget_4panel.gif"); FPS=6
tmp=[]
for k in range(len(all_fr)):
    p=str(REPO/f"_bp_{k:03d}.png"); show4(k,save=p); tmp.append(p)
imageio.mimsave(OUT,[imageio.imread(p) for p in tmp],duration=1.0/FPS,loop=0)
for p in tmp:
    try: os.remove(p)
    except OSError: pass
print("saved:",OUT)